# CT Corruption Detection - Cropping + Motion Blur
**Purpose:** Benchmark 2 new corruption types across ResNet-18, U-Net, EfficientNet.

### New Corruption Types:
- **Anatomical Cropping** - 5 sub-modes (top, bottom, both, center, random) - simulates incomplete scan coverage


In [ ]:
!pip -q install torch torchvision torchaudio

In [ ]:
import os
import glob
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt
from scipy.ndimage import zoom, gaussian_filter
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Dataset checking

In [ ]:
data_dir = "/kaggle/input/datasets/fardinabdullaacanto"
mask_dir = "/kaggle/input/datasets/gprem09/seg-lungs-luna16/seg-lungs-LUNA16"

all_files = sorted(glob.glob(os.path.join(data_dir, "**/*.mhd"), recursive=True))

scans = []
for p in all_files:
    if "seg-lungs-LUNA16" not in p:
        scans.append(p)

In [ ]:
print(len(scans))
print(scans[0])

In [ ]:
def load_img(path):
    img = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(img)
    return arr.astype(np.float32)

In [ ]:
def get_mask(scan_p, m_dir):
    fname = os.path.basename(scan_p)
    res = glob.glob(os.path.join(m_dir, "**", fname), recursive=True)
    if len(res) > 0:
        return load_img(res[0])
    return np.zeros_like(load_img(scan_p))

In [ ]:
test_path = scans[0]
raw_vol = load_img(test_path)
true_mask = get_mask(test_path, mask_dir)

In [ ]:
print(raw_vol.shape)

# 2. Preprocess

In [ ]:
def apply_mask(vol, m, bg=-1000.0):
    masked = np.copy(vol)
    masked[m == 0] = bg
    return masked

In [ ]:
def resize_data(vol, shape=(64, 128, 128), is_mask=False):
    curr = vol.shape
    f = [shape[0]/curr[0], shape[1]/curr[1], shape[2]/curr[2]]
    o = 0 if is_mask else 1
    return zoom(vol, f, order=o)

In [ ]:
def norm_data(vol, mn=-1000, mx=400):
    c = np.clip(vol, mn, mx)
    n = (c - mn) / (mx - mn)
    return n

In [ ]:
lungs = apply_mask(raw_vol, true_mask)
lungs_res = resize_data(lungs)
preped = norm_data(lungs_res)

In [ ]:
print(preped.shape)
print(preped.min(), preped.max())

In [ ]:
z = raw_vol.shape[0] // 2
z2 = preped.shape[0] // 2

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(raw_vol[z], cmap='gray', vmin=-1000, vmax=400)
ax[0].set_title("raw")
ax[1].imshow(lungs[z], cmap='gray', vmin=-1000, vmax=400)
ax[1].set_title("lungs")
ax[2].imshow(preped[z2], cmap='gray', vmin=0, vmax=1)
ax[2].set_title("resized + norm")
plt.show()

In [ ]:
c1 = raw_vol.shape[1] // 2
c2 = preped.shape[1] // 2

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(raw_vol[:, c1, :], cmap='gray', origin='lower', aspect='auto', vmin=-1000, vmax=400)
ax[0].set_title("raw (coronal)")
ax[1].imshow(lungs[:, c1, :], cmap='gray', origin='lower', aspect='auto', vmin=-1000, vmax=400)
ax[1].set_title("lungs (coronal)")
ax[2].imshow(preped[:, c2, :], cmap='gray', origin='lower', aspect='auto', vmin=0, vmax=1)
ax[2].set_title("resized + norm (coronal)")
plt.tight_layout()
plt.show()

# 3. Augmentation - Cropping & Blurring

In [ ]:
def apply_anatomical_cropping(preped_vol, res_mask, ratio=(0.15, 0.35), mode="random"):
    """
    Crop portions of the volume to simulate incomplete scan coverage.
    
    Parameters
    preped_vol : np.ndarray
        Preprocessed volume, shape (D, H, W), range [0, 1].
    res_mask : np.ndarray
        Resized lung mask, shape (D, H, W).
    ratio : tuple
        (min, max) fraction of slices to crop.
    mode : str
        One of "top", "bottom", "both", "center", "random".
        If "random", randomly chooses one of the 5 modes.
    
    """
    d, h, w = preped_vol.shape
    corrupt = np.copy(preped_vol)
    y = np.zeros_like(preped_vol, dtype=np.float32)
    
    r = np.random.uniform(ratio[0], ratio[1])
    n_remove = int(d * r)
    
    if mode == "random":
        mode = np.random.choice(["top", "bottom", "both", "center", "random_slices"])
    
    if mode == "top":
        slc = slice(0, n_remove)
        y[slc] = res_mask[slc]
        corrupt[slc] = 0.0
    
    elif mode == "bottom":
        slc = slice(d - n_remove, d)
        y[slc] = res_mask[slc]
        corrupt[slc] = 0.0
    
    elif mode == "both":
        n_top = n_remove // 2
        n_bot = n_remove - n_top
        slc_top = slice(0, n_top)
        slc_bot = slice(d - n_bot, d)
        y[slc_top] = res_mask[slc_top]
        y[slc_bot] = res_mask[slc_bot]
        corrupt[slc_top] = 0.0
        corrupt[slc_bot] = 0.0
    
    elif mode == "center":
        earliest_start = int(d * 0.25)
        latest_start = max(earliest_start, int(d * 0.75) - n_remove)
        start = np.random.randint(earliest_start, latest_start + 1)
        slc = slice(start, start + n_remove)
        y[slc] = res_mask[slc]
        corrupt[slc] = 0.0
    
    elif mode == "random_slices":
        remove_indices = np.random.choice(d, size=n_remove, replace=False)
        for idx in remove_indices:
            y[idx] = res_mask[idx]
            corrupt[idx] = 0.0
    
    return corrupt, y

In [ ]:
def apply_motion_blur(preped_vol, res_mask, kernel_size_range=(3, 9)):
    """
    Apply directional motion blur to simulate patient movement during scan.
    
    Parameters
    preped_vol : np.ndarray
        Preprocessed volume, shape (D, H, W), range [0, 1].
    res_mask : np.ndarray
        Resized lung mask, shape (D, H, W).
    kernel_size_range : tuple
        (min, max) kernel size for motion blur (must be odd). Typical: (3, 9).
        Smaller values = mild blur, larger = more pronounced motion.
    
    """
    kernel_size = np.random.choice(range(kernel_size_range[0], kernel_size_range[1] + 1, 2))
    
    direction = np.random.choice(['axial', 'coronal', 'sagittal'])
    
    if direction == 'axial':
        sigma_vec = (kernel_size / 6.0, 0.5, 0.5)
    elif direction == 'coronal':
        sigma_vec = (0.5, kernel_size / 6.0, 0.5)
    else:
        sigma_vec = (0.5, 0.5, kernel_size / 6.0)
    
    blurred = gaussian_filter(preped_vol, sigma=sigma_vec, mode='nearest')
    
    y = res_mask.copy()
    
    return blurred.astype(np.float32), y

In [ ]:
mask_res = resize_data(true_mask, is_mask=True)

x_crop, y_crop = apply_anatomical_cropping(preped, mask_res, mode="center")
x_mblur, y_mblur = apply_motion_blur(preped, mask_res)

In [ ]:
print("Cropping:", x_crop.shape, y_crop.shape)
print("Motion blur:", x_mblur.shape, y_mblur.shape)

# 4. Visualization of Augmentations

In [ ]:
z = preped.shape[0] // 2

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Anatomical Cropping - Center Mode", fontsize=14)

ax[0].imshow(preped[z], cmap='gray', vmin=0, vmax=1)
ax[0].set_title("Original")
ax[0].axis('off')

ax[1].imshow(x_crop[z], cmap='gray', vmin=0, vmax=1)
ax[1].set_title("Cropped")
ax[1].axis('off')

ax[2].imshow(y_crop[z], cmap='hot')
ax[2].set_title("Cropped Region Mask")
ax[2].axis('off')

ax[3].imshow(np.abs(preped[z] - x_crop[z]), cmap='hot')
ax[3].set_title("Absolute Difference")
ax[3].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle("Motion Blur (Directional)", fontsize=14)

ax[0].imshow(preped[z], cmap='gray', vmin=0, vmax=1)
ax[0].set_title("Original")
ax[0].axis('off')

ax[1].imshow(x_mblur[z], cmap='gray', vmin=0, vmax=1)
ax[1].set_title("Motion Blurred")
ax[1].axis('off')

ax[2].imshow(np.abs(preped[z] - x_mblur[z]), cmap='hot')
ax[2].set_title("Absolute Difference")
ax[2].axis('off')

plt.tight_layout()
plt.show()

# 5. Combo Augmentation

In [ ]:
def apply_combo_augmentation(preped_vol, res_mask, aug_types=["crop", "motion"], params=None):
    """
    Apply both augmentation types.
    
    Parameters
    preped_vol : np.ndarray
        Preprocessed volume.
    res_mask : np.ndarray
        Resized lung mask.
    aug_types : list[str]
        List of augmentation types to apply. Valid: "crop", "motion".
    params : dict, optional
        Parameters for each augmentation. If None, uses defaults.

    """
    if params is None:
        params = {}
    
    vol = np.copy(preped_vol)
    combined_mask = np.zeros_like(res_mask, dtype=np.float32)
    
    for aug_type in aug_types:
        if aug_type == "crop":
            crop_params = params.get("crop", {})
            ratio = crop_params.get("ratio", (0.15, 0.35))
            mode = crop_params.get("mode", "random")
            vol, mask = apply_anatomical_cropping(vol, res_mask, ratio=ratio, mode=mode)
            combined_mask = np.maximum(combined_mask, mask)
        
        elif aug_type == "motion":
            motion_params = params.get("motion", {})
            kernel_size_range = motion_params.get("kernel_size_range", (3, 9))
            vol, mask = apply_motion_blur(vol, res_mask, kernel_size_range=kernel_size_range)
            combined_mask = np.maximum(combined_mask, mask)
    
    return vol, combined_mask

In [ ]:
x_combo_2, y_combo_2 = apply_combo_augmentation(
    preped, mask_res, 
    aug_types=["crop", "motion"],
    params={"crop": {"mode": "center"}}
)

print("Combo (crop + motion):", x_combo_2.shape, y_combo_2.shape)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Combo Augmentations", fontsize=16)

ax[0, 0].imshow(preped[z], cmap='gray', vmin=0, vmax=1)
ax[0, 0].set_title("Original")
ax[0, 0].axis('off')

ax[0, 1].imshow(x_combo_2[z], cmap='gray', vmin=0, vmax=1)
ax[0, 1].set_title("Crop + Motion Blur")
ax[0, 1].axis('off')

ax[0, 2].imshow(np.abs(preped[z] - x_combo_2[z]), cmap='hot')
ax[0, 2].set_title("Difference (Combo)")
ax[0, 2].axis('off')

ax[1, 0].imshow(y_combo_2[z], cmap='hot')
ax[1, 0].set_title("Mask (Combo)")
ax[1, 0].axis('off')

ax[1, 1].imshow(x_combo_2[:, x_combo_2.shape[1]//2, :], cmap='gray', vmin=0, vmax=1, origin='lower', aspect='auto')
ax[1, 1].set_title("Coronal View")
ax[1, 1].axis('off')

ax[1, 2].imshow(y_combo_2[:, y_combo_2.shape[1]//2, :], cmap='hot', origin='lower', aspect='auto')
ax[1, 2].set_title("Coronal Mask")
ax[1, 2].axis('off')

plt.tight_layout()
plt.show()

# 6. Dataset Class with Augmentation

In [ ]:
class AugmentedCTDataset(Dataset):
    def __init__(self, scan_paths, mask_dir, target_shape=(64, 128, 128), 
                 augment=True, aug_prob=0.5):
        """
        Parameters
        scan_paths : list[str]
            List of .mhd file paths.
        mask_dir : str
            Directory containing lung masks.
        target_shape : tuple
            Target shape after preprocessing.
        augment : bool
            Whether to apply augmentation.
        aug_prob : float
            Probability of applying augmentation to each sample.
        """
        self.scan_paths = scan_paths
        self.mask_dir = mask_dir
        self.target_shape = target_shape
        self.augment = augment
        self.aug_prob = aug_prob
    
    def __len__(self):
        return len(self.scan_paths)
    
    def __getitem__(self, idx):
        scan_path = self.scan_paths[idx]
        
        raw_vol = load_img(scan_path)
        mask = get_mask(scan_path, self.mask_dir)
        
        lungs = apply_mask(raw_vol, mask)
        lungs_res = resize_data(lungs, shape=self.target_shape)
        preped = norm_data(lungs_res)
        mask_res = resize_data(mask, shape=self.target_shape, is_mask=True)
        
        if self.augment and np.random.rand() < self.aug_prob:
            aug_type = np.random.choice(["crop", "motion", "combo_2"])
            
            if aug_type == "crop":
                mode = np.random.choice(["top", "bottom", "both", "center", "random_slices"])
                x, y = apply_anatomical_cropping(preped, mask_res, mode=mode)
            elif aug_type == "motion":
                x, y = apply_motion_blur(preped, mask_res)
            else:
                x, y = apply_combo_augmentation(preped, mask_res, aug_types=["crop", "motion"])
        else:
            x = preped
            y = np.zeros_like(mask_res, dtype=np.float32)
        
        x = np.expand_dims(x, axis=0)
        y = np.expand_dims(y, axis=0)
        
        return torch.from_numpy(x), torch.from_numpy(y)

In [ ]:
train_size = int(0.7 * len(scans))
val_size = int(0.15 * len(scans))

train_scans = scans[:train_size]
val_scans = scans[train_size:train_size + val_size]
test_scans = scans[train_size + val_size:]

print(f"Train: {len(train_scans)}")
print(f"Val:   {len(val_scans)}")
print(f"Test:  {len(test_scans)}")

In [ ]:
train_dataset = AugmentedCTDataset(train_scans, mask_dir, augment=True, aug_prob=0.5)
val_dataset = AugmentedCTDataset(val_scans, mask_dir, augment=False)
test_dataset = AugmentedCTDataset(test_scans, mask_dir, augment=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)

print("DataLoaders created.")

In [ ]:
x_batch, y_batch = next(iter(train_loader))
print(f"Batch shapes: x={tuple(x_batch.shape)}, y={tuple(y_batch.shape)}")
print(f"x range: [{x_batch.min():.3f}, {x_batch.max():.3f}]")
print(f"y range: [{y_batch.min():.3f}, {y_batch.max():.3f}]")

# 7. Visualize Batch Examples

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle("Training Batch Samples (Augmented)", fontsize=16)

for i in range(min(4, x_batch.shape[0])):
    vol = x_batch[i, 0].numpy()
    mask = y_batch[i, 0].numpy()
    z = vol.shape[0] // 2
    
    axes[i, 0].imshow(vol[z], cmap='gray', vmin=0, vmax=1)
    axes[i, 0].set_title(f"Sample {i+1} - Volume (axial)")
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(mask[z], cmap='hot')
    axes[i, 1].set_title(f"Sample {i+1} - Mask (axial)")
    axes[i, 1].axis('off')
    
    c = vol.shape[1] // 2
    axes[i, 2].imshow(vol[:, c, :], cmap='gray', vmin=0, vmax=1, origin='lower', aspect='auto')
    axes[i, 2].set_title(f"Sample {i+1} - Volume (coronal)")
    axes[i, 2].axis('off')
    
    axes[i, 3].imshow(mask[:, c, :], cmap='hot', origin='lower', aspect='auto')
    axes[i, 3].set_title(f"Sample {i+1} - Mask (coronal)")
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()